In [172]:
import pandas as pd
import numpy as np

read=pd.read_csv('shipment_legs.csv')

In [173]:
read.info()
read.head()

<class 'pandas.DataFrame'>
RangeIndex: 27031 entries, 0 to 27030
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   LegID                  27031 non-null  str    
 1   ShipmentID             27031 non-null  str    
 2   LegSequence            27031 non-null  int64  
 3   CarrierID              27031 non-null  str    
 4   OriginPortID           27031 non-null  str    
 5   DestinationPortID      27031 non-null  str    
 6   Mode                   27031 non-null  str    
 7   DepartureTime          27031 non-null  str    
 8   ArrivalTime            27031 non-null  str    
 9   LegShippingCost_USD    27031 non-null  float64
 10  CustomsTariffPaid_USD  27031 non-null  float64
 11  DeclaredHSCode         27031 non-null  float64
 12  ClaimedOriginCountry   27031 non-null  str    
 13  DelayReason            27031 non-null  str    
dtypes: float64(3), int64(1), str(10)
memory usage: 5.3 MB


,LegID,ShipmentID,LegSequence,CarrierID,OriginPortID,DestinationPortID,Mode,DepartureTime,ArrivalTime,LegShippingCost_USD,CustomsTariffPaid_USD,DeclaredHSCode,ClaimedOriginCountry,DelayReason
0,LEG-SHP-100001-1,SHP-100001,1,C10,P01,P07,Ocean,2024-05-23 00:00:00,2024-05-31 04:17:26,413.86,121683.45,8517.12,China,On-time
1,LEG-SHP-100001-2,SHP-100001,2,C06,P07,P08,Rail,2024-05-31 09:17:26,2024-06-04 01:32:37,684.86,121683.45,8517.12,China,On-time
2,LEG-SHP-100001-3,SHP-100001,3,C08,P08,P12,Road,2024-06-04 13:32:37,2024-06-07 03:15:19,539.86,0.00,8517.18,Vietnam,On-time
3,LEG-SHP-100002-1,SHP-100002,1,C07,P06,P03,Air,2024-04-14 17:00:00,2024-04-16 22:50:36,755.87,18064.77,3004.90,Germany,On-time
4,LEG-SHP-100003-1,SHP-100003,1,C10,P01,P07,Ocean,2024-10-23 09:00:00,2024-11-03 02:17:27,654.03,33892.93,8471.30,China,On-time


In [174]:
date_columns = ['DepartureTime', 'ArrivalTime']
for col in date_columns:
    read[col] = pd.to_datetime(read[col], errors='coerce')

print(read["DepartureTime"].isna().sum())
print(read["ArrivalTime"].isna().sum())

#read["DeclaredHSCode"]=read["DeclaredHSCode"].astype(str)

0
0


In [175]:
read["DepartureDate"] = read["DepartureTime"].dt.normalize()
read["ArrivalDate"] = read["ArrivalTime"].dt.normalize()

read["DepartureYear"] = read["DepartureTime"].dt.year
read["DepartureMonth"] = read["DepartureTime"].dt.month
read["DepartureHour"] = read["DepartureTime"].dt.hour
read["DepartureDayOfWeek"] = read["DepartureTime"].dt.day_name()

In [176]:
read["LegDurationHours"] = (
    (read["ArrivalTime"] - read["DepartureTime"])
    .dt.total_seconds() / 3600
)

read["LegDurationDays"] = read["LegDurationHours"] / 24

In [177]:
read.info()

<class 'pandas.DataFrame'>
RangeIndex: 27031 entries, 0 to 27030
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   LegID                  27031 non-null  str           
 1   ShipmentID             27031 non-null  str           
 2   LegSequence            27031 non-null  int64         
 3   CarrierID              27031 non-null  str           
 4   OriginPortID           27031 non-null  str           
 5   DestinationPortID      27031 non-null  str           
 6   Mode                   27031 non-null  str           
 7   DepartureTime          27031 non-null  datetime64[us]
 8   ArrivalTime            27031 non-null  datetime64[us]
 9   LegShippingCost_USD    27031 non-null  float64       
 10  CustomsTariffPaid_USD  27031 non-null  float64       
 11  DeclaredHSCode         27031 non-null  float64       
 12  ClaimedOriginCountry   27031 non-null  str           
 13  DelayReason 

In [178]:
id_columns = [
    "LegID",
    "ShipmentID",
    "CarrierID",
    "OriginPortID",
    "DestinationPortID"
]

for col in id_columns:
    read[col] = read[col].astype(str).str.strip()

In [179]:
read.head(20)

,LegID,ShipmentID,LegSequence,CarrierID,OriginPortID,DestinationPortID,Mode,DepartureTime,ArrivalTime,LegShippingCost_USD,...,ClaimedOriginCountry,DelayReason,DepartureDate,ArrivalDate,DepartureYear,DepartureMonth,DepartureHour,DepartureDayOfWeek,LegDurationHours,LegDurationDays
0,LEG-SHP-100001-1,SHP-100001,1,C10,P01,P07,Ocean,2024-05-23 00:00:00,2024-05-31 04:17:26,413.86,...,China,On-time,2024-05-23,2024-05-31,2024,5,0,Thursday,196.290556,8.178773
1,LEG-SHP-100001-2,SHP-100001,2,C06,P07,P08,Rail,2024-05-31 09:17:26,2024-06-04 01:32:37,684.86,...,China,On-time,2024-05-31,2024-06-04,2024,5,9,Friday,88.253056,3.677211
2,LEG-SHP-100001-3,SHP-100001,3,C08,P08,P12,Road,2024-06-04 13:32:37,2024-06-07 03:15:19,539.86,...,Vietnam,On-time,2024-06-04,2024-06-07,2024,6,13,Tuesday,61.711667,2.571319
3,LEG-SHP-100002-1,SHP-100002,1,C07,P06,P03,Air,2024-04-14 17:00:00,2024-04-16 22:50:36,755.87,...,Germany,On-time,2024-04-14,2024-04-16,2024,4,17,Sunday,53.843333,2.243472
4,LEG-SHP-100003-1,SHP-100003,1,C10,P01,P07,Ocean,2024-10-23 09:00:00,2024-11-03 02:17:27,654.03,...,China,On-time,2024-10-23,2024-11-03,2024,10,9,Wednesday,257.290833,10.720451
5,LEG-SHP-100003-2,SHP-100003,2,C06,P07,P08,Rail,2024-11-03 09:17:27,2024-11-08 04:41:35,619.03,...,China,On-time,2024-11-03,2024-11-08,2024,11,9,Sunday,115.402222,4.808426
6,LEG-SHP-100003-3,SHP-100003,3,C09,P08,P04,Road,2024-11-08 16:41:35,2024-11-11 04:05:31,785.03,...,Vietnam,On-time,2024-11-08,2024-11-11,2024,11,16,Friday,59.398889,2.474954
7,LEG-SHP-100004-1,SHP-100004,1,C01,P06,P04,Ocean,2026-03-05 16:00:00,2026-03-18 22:11:48,1641.31,...,Germany,On-time,2026-03-05,2026-03-18,2026,3,16,Thursday,318.196667,13.258194
8,LEG-SHP-100005-1,SHP-100005,1,C10,P06,P04,Ocean,2025-07-29 00:00:00,2025-08-09 01:22:18,861.25,...,Germany,On-time,2025-07-29,2025-08-09,2025,7,0,Tuesday,265.371667,11.057153
9,LEG-SHP-100005-2,SHP-100005,2,C09,P03,P12,Rail,2025-08-09 04:22:18,2025-08-12 15:35:49,965.25,...,Germany,On-time,2025-08-09,2025-08-12,2025,8,4,Saturday,83.225278,3.467720


In [180]:
extracted = read["LegID"].str.extract(r"^LEG-(SHP-\d+)-(\d+)$")

read["LegIDShipmentID"] = extracted[0]
read["LegIDSequence"] = pd.to_numeric(extracted[1], errors="coerce")

read["LegIDMismatchFlag"] = (
    (read["LegIDShipmentID"] != read["ShipmentID"]) |
    (read["LegIDSequence"] != read["LegSequence"])
)

In [181]:
print(read["LegIDMismatchFlag"].sum())

0


In [182]:
print(read["LegID"].duplicated().sum())

0


In [183]:
duplicate_sequence_count = (
    read.groupby("ShipmentID")["LegSequence"]
    .apply(lambda x: x.duplicated().sum())
    .sum()
)

print(duplicate_sequence_count)

0


In [184]:
read["ArrivalBeforeDepartureFlag"] = (
    read["ArrivalTime"] < read["DepartureTime"]
)

In [185]:
today = pd.Timestamp.today().normalize()

read["FutureDepartureFlag"] = read["DepartureTime"] > today
read["FutureArrivalFlag"] = read["ArrivalTime"] > today

In [186]:
def clean_hs_code(value):
    s = str(value).strip()

    if s.lower() in ["nan", "none", "<na>", ""]:
        return ""

    # If format is like 8517.12 or 306.17
    if "." in s:
        left, right = s.split(".", 1)

        # HS heading should usually be 4 digits
        left = left.zfill(4)

        # HS subheading should usually be 2 digits
        right = (right + "00")[:2]

        return (left + right).zfill(6)

    # If already numeric string
    if s.isdigit():
        return s.zfill(6)

    return s
read["HSCode6"] = read["DeclaredHSCode"].apply(clean_hs_code)

In [187]:
read["HSCodeChapter"] = read["HSCode6"].str[:2]
read["HSCodeHeading"] = read["HSCode6"].str[:4]

In [188]:
read["InvalidHSCodeFlag"] = (
    ~read["HSCode6"].str.match(r"^\d{6,12}$")
)
print(read["InvalidHSCodeFlag"].sum())

0


In [189]:
read["NegativeLegCostFlag"] = read["LegShippingCost_USD"] < 0
read["NegativeLegCustomsFlag"] = read["CustomsTariffPaid_USD"] < 0
read["ZeroLegCostFlag"] = read["LegShippingCost_USD"] == 0
read["HasLegCustomsFlag"] = read["CustomsTariffPaid_USD"] > 0

print(read["NegativeLegCostFlag"].sum())
print(read["NegativeLegCustomsFlag"].sum())
print(read["ZeroLegCostFlag"].sum())

0
0
0


In [190]:
read = read.sort_values(["ShipmentID", "LegSequence"]).reset_index(drop=True)

In [191]:
read["PreviousDestinationPortID"] = (
    read.groupby("ShipmentID")["DestinationPortID"]
    .shift(1)
)

read["RouteBreakFlag"] = (
    (read["LegSequence"] > 1) &
    (read["OriginPortID"] != read["PreviousDestinationPortID"])
)
print(read["RouteBreakFlag"].sum())

4858


In [192]:
read["PreviousArrivalTime"] = (
    read.groupby("ShipmentID")["ArrivalTime"]
    .shift(1)
)

read["GapHours"] = (
    (read["DepartureTime"] - read["PreviousArrivalTime"])
    .dt.total_seconds() / 3600
)

read["NegativeGapFlag"] = read["GapHours"] < 0
read["LongGapFlag"] = read["GapHours"] > 48

In [193]:
read["DepartureBeforePreviousArrivalFlag"] = (
    (read["LegSequence"] > 1) &
    (read["DepartureTime"] < read["PreviousArrivalTime"])
)

In [194]:
read["SameOriginDestinationFlag"] = (
    read["OriginPortID"] == read["DestinationPortID"]
)

In [195]:
hs_count = (
    read.groupby("ShipmentID")["HSCode6"]
    .nunique()
    .reset_index()
)

hs_count.columns = ["ShipmentID", "HSCodeCount_x"]

read = read.merge(hs_count, on="ShipmentID", how="left")

read["HSCodeInconsistentFlag"] = read["HSCodeCount_x"] > 1

In [196]:
country_count = (
    read.groupby("ShipmentID")["ClaimedOriginCountry"]
    .nunique()
    .reset_index()
)

country_count.columns = ["ShipmentID", "ClaimedCountryCount"]

read = read.merge(country_count, on="ShipmentID", how="left")

read["ClaimedCountryInconsistentFlag"] = read["ClaimedCountryCount"] > 1

In [197]:
read["Route"] = (
    read["OriginPortID"] + " → " + read["DestinationPortID"]
)

In [198]:
read["DelayReason"] = read["DelayReason"].astype(str).str.strip()

no_delay_values = [
    "on-time",
    "on time",
    "no delay",
    "none",
    "na",
    "n/a",
    "no",
    "ok",
    "Typhoon Force Majeure"
]

read["HasDelayReason"] = (
    ~read["DelayReason"].str.lower().isin(no_delay_values)
)
print(read["HasDelayReason"].value_counts())

HasDelayReason
False    26066
True       965
Name: count, dtype: int64


In [199]:
read.head()

,LegID,ShipmentID,LegSequence,CarrierID,OriginPortID,DestinationPortID,Mode,DepartureTime,ArrivalTime,LegShippingCost_USD,...,NegativeGapFlag,LongGapFlag,DepartureBeforePreviousArrivalFlag,SameOriginDestinationFlag,HSCodeCount_x,HSCodeInconsistentFlag,ClaimedCountryCount,ClaimedCountryInconsistentFlag,Route,HasDelayReason
0,LEG-SHP-100001-1,SHP-100001,1,C10,P01,P07,Ocean,2024-05-23 00:00:00,2024-05-31 04:17:26,413.86,...,False,False,False,False,2,True,2,True,P01 → P07,False
1,LEG-SHP-100001-2,SHP-100001,2,C06,P07,P08,Rail,2024-05-31 09:17:26,2024-06-04 01:32:37,684.86,...,False,False,False,False,2,True,2,True,P07 → P08,False
2,LEG-SHP-100001-3,SHP-100001,3,C08,P08,P12,Road,2024-06-04 13:32:37,2024-06-07 03:15:19,539.86,...,False,False,False,False,2,True,2,True,P08 → P12,False
3,LEG-SHP-100002-1,SHP-100002,1,C07,P06,P03,Air,2024-04-14 17:00:00,2024-04-16 22:50:36,755.87,...,False,False,False,False,1,False,1,False,P06 → P03,False
4,LEG-SHP-100003-1,SHP-100003,1,C10,P01,P07,Ocean,2024-10-23 09:00:00,2024-11-03 02:17:27,654.03,...,False,False,False,False,1,False,2,True,P01 → P07,False


In [200]:
shipment=pd.read_csv('shipments.csv')
valid_shipments = set(shipment["ShipmentID"])

read["OrphanShipmentIDFlag"] = ~read["ShipmentID"].isin(valid_shipments)

print(read["OrphanShipmentIDFlag"].sum())

0


In [201]:
leg_summary = read.groupby("ShipmentID").agg(
    LegCount=("LegID", "count"),
    TotalLegShippingCost_USD=("LegShippingCost_USD", "sum"),
    TotalLegCustomsPaid_USD=("CustomsTariffPaid_USD", "sum"),
    FirstLegDepartureTime=("DepartureTime", "min"),
    LastLegArrivalTime=("ArrivalTime", "max"),
    FirstOriginPortID=("OriginPortID", "first"),
    FinalDestinationPortID=("DestinationPortID", "last"),
    ModeCount=("Mode", "nunique"),
    HasLegDelayReason=("HasDelayReason", "max"),
    RouteBreakCount=("RouteBreakFlag", "sum"),
    DepartureBeforePreviousArrivalCount=("DepartureBeforePreviousArrivalFlag", "sum"),
    FutureArrivalLegCount=("FutureArrivalFlag", "sum"),
    HSCodeInconsistentFlag=("HSCodeInconsistentFlag", "max"),
    ClaimedCountryInconsistentFlag=("ClaimedCountryInconsistentFlag", "max")
).reset_index()

leg_summary["IsMultimodal"] = leg_summary["ModeCount"] > 1

shipment = shipment.merge(
    leg_summary,
    on="ShipmentID",
    how="left"
)

shipment["LegCount"] = shipment["LegCount"].fillna(0).astype(int)
shipment["TotalLegShippingCost_USD"] = shipment["TotalLegShippingCost_USD"].fillna(0)
shipment["TotalLegCustomsPaid_USD"] = shipment["TotalLegCustomsPaid_USD"].fillna(0)
shipment["HasLegDelayReason"] = shipment["HasLegDelayReason"].fillna(False)
shipment["RouteBreakCount"] = shipment["RouteBreakCount"].fillna(0).astype(int)

shipment["NoLegsFlag"] = shipment["LegCount"] == 0

In [ ]:
shipment.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 44 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   ShipmentID                             15000 non-null  str           
 1   CargoType                              15000 non-null  str           
 2   StorageRequirement                     15000 non-null  str           
 3   TotalWeight_KG                         15000 non-null  float64       
 4   DeclaredValue_USD                      15000 non-null  float64       
 5   BookingDate                            15000 non-null  str           
 6   ScheduledDeliveryDate                  15000 non-null  str           
 7   ActualDeliveryDate                     15000 non-null  str           
 8   FinalStatus                            15000 non-null  str           
 9   TotalShippingCost_USD                  15000 non-null  float64       
 1

In [202]:
shipment = shipment.merge(
    leg_summary,
    on="ShipmentID",
    how="left"
)

In [207]:
shipment["LegCount_x"] = shipment["LegCount_x"].fillna(0).astype(int)
shipment["TotalLegShippingCost_USD_x"] = shipment["TotalLegShippingCost_USD_x"].fillna(0)
shipment["TotalLegCustomsPaid_USD_x"] = shipment["TotalLegCustomsPaid_USD_x"].fillna(0)
shipment["HasLegDelayReason_x"] = shipment["HasLegDelayReason_x"].fillna(False)
shipment["RouteBreakCount_x"] = shipment["RouteBreakCount_x"].fillna(0).astype(int)

In [211]:
shipment["NoLegsFlag_x"] = shipment["LegCount_x"] == 0

In [210]:
shipment["LegCount_y"] = shipment["LegCount_y"].fillna(0).astype(int)
shipment["TotalLegShippingCost_USD_y"] = shipment["TotalLegShippingCost_USD_y"].fillna(0)
shipment["TotalLegCustomsPaid_USD_y"] = shipment["TotalLegCustomsPaid_USD_y"].fillna(0)
shipment["HasLegDelayReason_y"] = shipment["HasLegDelayReason_y"].fillna(False)
shipment["RouteBreakCount_y"] = shipment["RouteBreakCount_y"].fillna(0).astype(int)

In [212]:
shipment["NoLegsFlag_y"] = shipment["LegCount_y"] == 0

In [ ]:
shipment["ShippingCostMismatchFlag_x"] = (
    abs(
        shipment["TotalShippingCost_USD"] -
        shipment["TotalLegShippingCost_USD_x"]
    ) > 1
)
shipment["ShippingCostMismatchFlag_y"] = (
    abs(
        shipment["TotalShippingCost_USD"] -
        shipment["TotalLegShippingCost_USD_y"]
    ) > 1
)

In [215]:
shipment["LegCustomsMismatchFlag_x"] = (
    abs(
        shipment["TotalCustomsPaid_USD"] -
        shipment["TotalLegCustomsPaid_USD_x"]
    ) > 1
)
shipment["LegCustomsMismatchFlag_y"] = (
    abs(
        shipment["TotalCustomsPaid_USD"] -
        shipment["TotalLegCustomsPaid_USD_y"]
    ) > 1
)

In [217]:
shipment["TransshipmentMismatchFlag_x"] = (
    ((shipment["IsTransshipment"] == "Yes") & (shipment["LegCount_x"] <= 1)) |
    ((shipment["IsTransshipment"] == "No") & (shipment["LegCount_x"] > 1))
)
shipment["TransshipmentMismatchFlag_y"] = (
    ((shipment["IsTransshipment"] == "Yes") & (shipment["LegCount_y"] <= 1)) |
    ((shipment["IsTransshipment"] == "No") & (shipment["LegCount_y"] > 1))
)

In [218]:
print(shipment["TransshipmentMismatchFlag_x"].sum())
print(shipment["TransshipmentMismatchFlag_y"].sum())

9675
9675


In [222]:
shipment.head()

,ShipmentID,CargoType,StorageRequirement,TotalWeight_KG,DeclaredValue_USD,BookingDate,ScheduledDeliveryDate,ActualDeliveryDate,FinalStatus,TotalShippingCost_USD,...,HSCodeInconsistentFlag_y,ClaimedCountryInconsistentFlag_y,IsMultimodal_y,NoLegsFlag_x,NoLegsFlag_y,ShippingCostMismatchFlag_x,LegCustomsMismatchFlag_x,LegCustomsMismatchFlag_y,TransshipmentMismatchFlag_x,TransshipmentMismatchFlag_y
0,SHP-100001,Smartphones,Dry,2638.17,486733.78,2024-05-22,2024-06-08,2024-06-07,Delivered,1638.58,...,True,True,True,False,False,False,False,False,False,False
1,SHP-100002,Pharmaceuticals,Refrigerated,1426.08,903238.59,2024-04-14,2024-04-21,2024-04-17,Delivered,755.87,...,False,False,False,False,False,False,False,False,False,False
2,SHP-100003,Electronics,Dry,4160.39,338929.32,2024-10-23,2024-11-10,2024-11-11,Delivered,2058.09,...,False,True,True,False,False,False,False,False,False,False
3,SHP-100004,Auto Parts,Dry,15350.76,207514.00,2026-03-05,2026-03-12,2026-03-19,Delivered,1641.31,...,False,False,False,False,False,False,False,False,False,False
4,SHP-100005,Frozen Seafood,Refrigerated,7883.33,116878.59,2025-07-28,2025-08-12,2025-08-12,Delivered,1826.50,...,False,False,True,False,False,False,False,False,True,True


In [225]:
shipment["ShipmentLateNoLegDelayReasonFlag_x"] = (
    (shipment["HasLegDelayReason_x"] == True) &
    (shipment["HasLegDelayReason_x"] == False)
)
shipment["ShipmentLateNoLegDelayReasonFlag_y"] = (
    (shipment["HasLegDelayReason_y"] == True) &
    (shipment["HasLegDelayReason_y"] == False)
)

In [230]:
shipment["RouteBreakShipmentFlag_x"] = shipment["RouteBreakCount_x"] > 0
shipment["RouteBreakShipmentFlag_y"] = shipment["RouteBreakCount_y"] > 0

In [231]:
shipment.to_csv('shpiment_cleaned2.csv', index=False)
read.to_csv('read_cleaned.csv', index=False)